In [ ]:
# ============================================================
# Elevation Extraction from DEM Data (.TIF)
# ============================================================

# ── Cell 1: Install (run once) ─────────────────────
!apt-get install -y gdal-bin python3-gdal -q
!pip install rasterio geopandas shapely psutil -q

import os
import glob
import subprocess
import numpy as np
import rasterio
import geopandas as gpd
from shapely.geometry import LineString, MultiPolygon, Polygon
from shapely.ops import polygonize, unary_union
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings("ignore")

# Ensure plots are displayed inline in Colab
%matplotlib inline

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


# ============================================================
# CONFIGURATION ← Edit these before running
# ============================================================
INPUT_FOLDER      = "/content/drive/MyDrive/DEM"   # DEM folder
OUTPUT_FOLDER     = "/content/drive/MyDrive/dem_output" # Changed to save to Google Drive
CONTOUR_INTERVAL  = 1.0                                # metres

# Output paths (auto-generated)
MERGED_DEM        = os.path.join(OUTPUT_FOLDER, "merged_dem.tif")
LINE_SHP          = os.path.join(OUTPUT_FOLDER, "elevation_contours_lines.shp")
POLY_SHP          = os.path.join(OUTPUT_FOLDER, "elevation_contours_polygons.shp")
POLY_GEOJSON      = os.path.join(OUTPUT_FOLDER, "elevation_contours_polygons.geojson")
# ============================================================


# ── STEP 1: Setup ─────────────────────────
def setup():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    print(f"[✓] Output folder: {OUTPUT_FOLDER}")


# ── STEP 2: Find DEM files ─────────────────────
def find_dem_files():
    exts = ["*.tif", "*.tiff", "*.img", "*.asc", "*.dem"]
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(INPUT_FOLDER, "**", ext), recursive=True))
        files.extend(glob.glob(os.path.join(INPUT_FOLDER, ext)))
    files = sorted(list(set(files)))
    if not files:
        raise FileNotFoundError(f"No DEM files found in: {INPUT_FOLDER}")
    print(f"[✓] Found {len(files)} DEM file(s):")
    for f in files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"    • {os.path.basename(f):30s}  ({size_mb:.1f} MB)")
    return files


# ── STEP 3: RAM check ─────────────────────────
def check_memory():
    import psutil
    ram = psutil.virtual_memory()
    print(f"\n[i] Available RAM: {ram.available/1024**3:.1f} GB / {ram.total/1024**3:.1f} GB total")


# ── STEP 4: Merge DEMs with GDAL ──────────────────
def get_nodata_value(dem_file):
    """Read the nodata value from a DEM file."""
    info = subprocess.run(["gdalinfo", dem_file], capture_output=True, text=True)
    for line in info.stdout.split("\n"):
        if "NoData Value" in line:
            try:
                val = float(line.split("=")[1].strip())
                print(f"    Detected nodata value: {val}")
                return val
            except:
                pass
    return None


def merge_with_gdal(dem_files):
    print(f"\n[⏳] Merging {len(dem_files)} DEM files (GDAL, disk-based)...")

    # Detect nodata from first file
    nodata_val = get_nodata_value(dem_files[0])

    # Remove existing output to avoid conflicts
    if os.path.exists(MERGED_DEM):
        os.remove(MERGED_DEM)

    cmd = [
        "gdalwarp",
        "-of", "GTiff",
        "-co", "COMPRESS=LZW",
        "-co", "TILED=YES",
        "-co", "BIGTIFF=YES",
        "-r", "nearest",
        "-multi",
        "-wm", "512",
        "--config", "GDAL_CACHEMAX", "512",
        "-overwrite",
    ]

    # Explicitly set nodata so gdalwarp handles -FLT_MAX correctly
    if nodata_val is not None:
        cmd += ["-srcnodata", str(nodata_val), "-dstnodata", "-9999"]

    cmd += dem_files + [MERGED_DEM]

    print(f"    Running gdalwarp...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-500:] if result.stdout else "")

    if result.returncode != 0 or not os.path.exists(MERGED_DEM):
        print(f"[!] gdalwarp failed:\n{result.stderr[-1000:]}")
        print("[i] Trying gdal_merge.py fallback...")

        nodata_arg = str(nodata_val) if nodata_val is not None else "-3.40282e+38"
        cmd2 = [
            "gdal_merge.py",
            "-o", MERGED_DEM,
            "-of", "GTiff",
            "-co", "COMPRESS=LZW",
            "-co", "BIGTIFF=YES",
            "-n", nodata_arg,      # input nodata to ignore
            "-a_nodata", "-9999",  # output nodata value
        ] + dem_files
        result2 = subprocess.run(cmd2, capture_output=True, text=True)
        print(result2.stdout[-500:] if result2.stdout else "")
        if result2.returncode != 0 or not os.path.exists(MERGED_DEM):
            raise RuntimeError(f"Both merge methods failed.\ngdalwarp: {result.stderr}\ngdal_merge: {result2.stderr}")

    size_mb = os.path.getsize(MERGED_DEM) / (1024 * 1024)
    print(f"[✓] Merged DEM saved: {MERGED_DEM}  ({size_mb:.1f} MB)")

    info = subprocess.run(["gdalinfo", "-mm", MERGED_DEM], capture_output=True, text=True)
    for line in info.stdout.split("\n"):
        if any(k in line for k in ["Size is", "Pixel Size", "Computed Min", "NoData"]):
            print(f"    {line.strip()}")


# ── STEP 5: Extract contour LINES with GDAL ─────────────────
def extract_contour_lines():
    print(f"\n[⏳] Extracting contour lines at {CONTOUR_INTERVAL}m interval...")

    cmd = [
        "gdal_contour",
        "-b", "1",
        "-a", "elevation",
        "-i", str(CONTOUR_INTERVAL),
        "-f", "ESRI Shapefile",
        MERGED_DEM,
        LINE_SHP
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"gdal_contour failed:\n{result.stderr}")

    gdf = gpd.read_file(LINE_SHP)
    print(f"[✓] Raw contour lines extracted: {len(gdf)} features")
    print(f"    Elevation range (raw): {gdf['elevation'].min():.1f}m — {gdf['elevation'].max():.1f}m")
    return gdf


# ── STEP 6: Remove negative elevations ──────────────────
def remove_negative_elevations(gdf):
    """Drop all contour lines where elevation < 0."""
    before = len(gdf)
    gdf_clean = gdf[gdf["elevation"] >= 0].copy()
    gdf_clean = gdf_clean.reset_index(drop=True)
    removed = before - len(gdf_clean)

    print(f"\n[✓] Negative elevation removal:")
    print(f"    Removed : {removed} features (elevation < 0)")
    print(f"    Kept    : {len(gdf_clean)} features")
    if len(gdf_clean) > 0:
        print(f"    Range   : {gdf_clean['elevation'].min():.1f}m — {gdf_clean['elevation'].max():.1f}m")

    # Overwrite the line shapefile with the cleaned version
    gdf_clean.to_file(LINE_SHP)
    print(f"[✓] Cleaned line shapefile saved: {LINE_SHP}")
    return gdf_clean


# ── STEP 7: Convert contour lines → polygons ───────────
def lines_to_polygons(gdf_lines):
    """
    Convert closed/open contour lines into filled elevation band polygons.

    Strategy:
      • For each unique elevation level, collect all line geometries.
      • Use shapely.ops.polygonize() to build polygons from closed rings.
      • For lines that don't close perfectly, buffer by a tiny amount to
        force closure, then polygonize.
      • Assign min/max elevation band attributes.
    """
    print(f"\n[⏳] Converting contour lines to polygons...")

    crs = gdf_lines.crs
    levels = sorted(gdf_lines["elevation"].unique())
    print(f"    Processing {len(levels)} elevation levels...")

    all_polys   = []
    all_elev    = []
    all_elev_min = []
    all_elev_max = []

    for i, elev in enumerate(levels):
        # Get all lines at this elevation
        lines = gdf_lines[gdf_lines["elevation"] == elev].geometry.tolist()

        # Merge all line segments at this level into a unified geometry
        merged = unary_union(lines)

        # Polygonize (works on closed rings)
        polys = list(polygonize(merged))

        # If polygonize produced nothing, try buffering lines slightly to close gaps
        if not polys:
            buffered = merged.buffer(0.01)   # 1cm buffer to close tiny gaps
            polys = list(polygonize(buffered.boundary))

        for poly in polys:
            if poly.is_valid and not poly.is_empty and poly.area > 0:
                all_polys.append(poly)
                all_elev.append(float(elev))
                # Elevation band: this level to next level
                next_elev = levels[i + 1] if i + 1 < len(levels) else elev + CONTOUR_INTERVAL
                all_elev_min.append(float(elev))
                all_elev_max.append(float(next_elev))

    if not all_polys:
        print("[!] WARNING: No closed polygons could be formed from contour lines.")
        print("    This can happen when contour lines don't form closed rings")
        print("    (e.g., lines cut by the DEM boundary). Trying convex hull approach...")
        all_polys, all_elev, all_elev_min, all_elev_max = lines_to_polygons_fallback(gdf_lines, levels)

    gdf_poly = gpd.GeoDataFrame(
        {
            "elevation" : all_elev,
            "elev_min"  : all_elev_min,
            "elev_max"  : all_elev_max,
            "elev_ft"   : [round(e * 3.28084, 2) for e in all_elev],
            "area_m2"   : [p.area for p in all_polys],
        },
        geometry=all_polys,
        crs=crs
    )

    # Fix any invalid geometries
    gdf_poly["geometry"] = gdf_poly["geometry"].buffer(0)
    gdf_poly = gdf_poly[gdf_poly.geometry.is_valid & ~gdf_poly.geometry.is_empty]
    gdf_poly = gdf_poly.reset_index(drop=True)

    print(f"[✓] Polygon conversion complete:")
    print(f"    Total polygons  : {len(gdf_poly)}")
    if len(gdf_poly) > 0:
        print(f"    Elevation range : {gdf_poly['elevation'].min():.1f}m — {gdf_poly['elevation'].max():.1f}m")

    # Save polygon shapefile
    gdf_poly.to_file(POLY_SHP)
    print(f"[✓] Polygon shapefile saved: {POLY_SHP}")

    # Save GeoJSON
    gdf_poly.to_file(POLY_GEOJSON, driver="GeoJSON")
    print(f"[✓] Polygon GeoJSON saved: {POLY_GEOJSON}")

    return gdf_poly


def lines_to_polygons_fallback(gdf_lines, levels):
    """
    Fallback: buffer each contour line slightly to create thin polygon strips
    representing each elevation band. Used when lines don't form closed rings.
    """
    print("    Using buffer-based polygon approach...")
    all_polys, all_elev, all_elev_min, all_elev_max = [], [], [], []

    for i, elev in enumerate(levels):
        lines = gdf_lines[gdf_lines["elevation"] == elev].geometry.tolist()
        merged = unary_union(lines)
        # Buffer by half the contour interval to create elevation band strips
        poly = merged.buffer(CONTOUR_INTERVAL / 2.0)
        if poly.is_valid and not poly.is_empty:
            next_elev = levels[i + 1] if i + 1 < len(levels) else elev + CONTOUR_INTERVAL
            all_polys.append(poly)
            all_elev.append(float(elev))
            all_elev_min.append(float(elev))
            all_elev_max.append(float(next_elev))

    return all_polys, all_elev, all_elev_min, all_elev_max


# ── STEP 8: Visualization ─────────────────────────
def plot_results(gdf_lines, gdf_poly):
    print("\n[⏳] Generating visualization...")
    try:
        fig, axes = plt.subplots(1, 3, figsize=(22, 7))
        print("    Figure and axes created.")

        # Left: Merged DEM overview
        with rasterio.open(MERGED_DEM) as src:
            scale = 10
            elev = src.read(
                1,
                out_shape=(1, max(1, src.height // scale), max(1, src.width // scale)),
                resampling=rasterio.enums.Resampling.average
            ).astype(np.float32)
            nodata = src.nodata
        if nodata is not None:
            elev[elev == nodata] = np.nan
        elev[elev < 0] = np.nan  # also mask negatives in preview
        print("    DEM data loaded for overview.")

        ax1 = axes[0]
        im = ax1.imshow(elev, cmap="terrain", interpolation="bilinear", aspect="equal")
        plt.colorbar(im, ax=ax1, label="Elevation (m)", shrink=0.8)
        ax1.set_title("Merged DEM", fontsize=12, fontweight="bold")
        ax1.axis("off")
        print("    Merged DEM plot generated.")

        # Middle: Cleaned contour lines
        ax2 = axes[1]
        # Reduce sample size for plotting to prevent memory issues with very large datasets
        sample_lines = gdf_lines.sample(min(len(gdf_lines), 20000)) if not gdf_lines.empty else gpd.GeoDataFrame()
        print(f"    Plotting {len(sample_lines)} sample contour lines.")
        if not sample_lines.empty:
            sample_lines.plot(ax=ax2, column="elevation", cmap="terrain",
                            linewidth=0.3, legend=True,
                            legend_kwds={"label": "Elevation (m)", "shrink": 0.8})
        ax2.set_title(f"Contour Lines @ {CONTOUR_INTERVAL}m\n(negatives removed)", fontsize=12, fontweight="bold")
        ax2.set_xlabel("Easting"); ax2.set_ylabel("Northing")
        print("    Contour lines plot generated.")

        # Right: Elevation polygons
        ax3 = axes[2]
        sample_poly = gdf_poly.sample(min(len(gdf_poly), 10000)) if not gdf_poly.empty else gpd.GeoDataFrame()
        print(f"    Plotting {len(sample_poly)} sample elevation polygons.")
        if not sample_poly.empty:
            sample_poly.plot(ax=ax3, column="elevation", cmap="terrain",
                            legend=True, alpha=0.8,
                            legend_kwds={"label": "Elevation (m)", "shrink": 0.8})
            ax3.set_title("Elevation Polygons", fontsize=12, fontweight="bold")
        else:
            ax3.text(0.5, 0.5, "No closed polygons\nformed from contours",
                     ha="center", va="center", transform=ax3.transAxes, fontsize=11)
            ax3.set_title("Elevation Polygons", fontsize=12, fontweight="bold")
        ax3.set_xlabel("Easting"); ax3.set_ylabel("Northing")
        print("    Elevation polygons plot generated.")

        plt.tight_layout()
        preview_path = os.path.join(OUTPUT_FOLDER, "dem_preview.png")
        print(f"    Attempting to save plot to: {preview_path}")
        plt.savefig(preview_path, dpi=150, bbox_inches="tight")
        print(f"[✓] Preview saved: {preview_path}")
        plt.show()
        plt.close(fig) # Explicitly close the figure to free memory
        print("    Figure closed.")
    except Exception as e:
        print(f"[!] Error during visualization: {e}")
        print("    Skipping visualization due to error.")


# ── STEP 9: Summary & Download ─────────────────────
def summary_and_download():
    print("\n" + "="*60)
    print("  OUTPUT FILES")
    print("="*60)
    for fname in sorted(os.listdir(OUTPUT_FOLDER)):
        fpath = os.path.join(OUTPUT_FOLDER, fname)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  ð {fname:48s} ({size_mb:.2f} MB)")
    print("="*60)

    try:
        from google.colab import files
        print("\n[⬇] Downloading output files...")
        for fname in sorted(os.listdir(OUTPUT_FOLDER)):
            fpath = os.path.join(OUTPUT_FOLDER, fname)
            files.download(fpath)
    except ImportError:
        print(f"\n[i] Files saved at: {OUTPUT_FOLDER}")


# ── MAIN ─────────────────────────
def main():
    print("=" * 60)
    print("  ALTM DEM Elevation Extractor  v3")
    print("  (Memory-Optimized + Negative Removal + Line→Polygon)")
    print("=" * 60)

    setup()
    dem_files = find_dem_files()
    check_memory()
    merge_with_gdal(dem_files)
    gdf_lines = extract_contour_lines()
    gdf_lines = remove_negative_elevations(gdf_lines)   # ← NEW: remove negatives
    gdf_poly  = lines_to_polygons(gdf_lines)             # ← NEW: line → polygon
    # plot_results(gdf_lines, gdf_poly) # Commented out visualization step
    summary_and_download()

    print("\n[✓] All done!")


main()